In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch04-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "04-01-molecular-descriptors.py"

從 ESOL 取三個分子式相同的分子，對照描述符與實測水溶解度。初次執行需連網下載資料。 練習可接在下方寫入 Python 腳本的儲存格末尾，再執行最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile

import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, rdMolDescriptors

np.random.seed(42)
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
tasks, splits, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], data_dir=str(cache), reload=False)
assert sum(map(len, splits)) == 1128
assert tasks == ["measured log solubility in mols per litre"]

# 僅取指定的 ESOL 紀錄，保留原始 SMILES 及未標準化的標籤。
names = {"CCCCO": "1-Butanol", "CCC(C)O": "2-Butanol", "CCOCC": "Diethyl ether"}
records = [(str(s).strip(), float(y[0])) for split in splits
           for s, y in zip(split.ids, split.y)]
selected = {s: y for s, y in records if s in names}
assert sum(s in names for s, _ in records) == len(selected) == len(names)

def oxygen_reach(mol):
    oxygen, = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() == 8]
    distances = Chem.GetDistanceMatrix(mol)
    return int(distances[oxygen].max())

rows = []
mols = []
for smiles, name in names.items():
    mol = Chem.MolFromSmiles(smiles)
    mols.append(mol)
    row = [rdMolDescriptors.CalcMolFormula(mol), f"{Descriptors.MolWt(mol):.3f}",
           str(rdMolDescriptors.CalcNumHBD(mol)), str(oxygen_reach(mol)),
           f"{selected[smiles]:.2f}"]
    rows.append(row)
    print(name, *row)

fig = plt.figure(figsize=(8.5, 4.4), layout="constrained")
grid = fig.add_gridspec(2, 3, height_ratios=[1.2, 1])
for i, (mol, name) in enumerate(zip(mols, names.values())):
    ax = fig.add_subplot(grid[0, i])
    ax.imshow(Draw.MolToImage(mol, size=(350, 170)))
    ax.set_title(name, fontsize=12)
    ax.axis("off")
ax = fig.add_subplot(grid[1, :])
ax.axis("off")
labels = ["Molecular formula", "Molar mass (g/mol)", "H-bond donors",
          "Farthest atom from O (bonds)", "Measured log S"]
table = ax.table(cellText=list(zip(*rows)), rowLabels=labels, colLabels=list(names.values()), cellLoc="center",
                 bbox=[0.40, 0.02, 0.58, 0.94])
table.auto_set_font_size(False)
table.set_fontsize(10)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor("white")
    cell.set_facecolor("#f1d9ba" if r == 5 else "#e1eff0")
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / "04-01-molecular-descriptors.png", dpi=160)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "04-01-molecular-descriptors.png")))